# B — VAE couleur
**Amjad AAYD et [information masquée] — [information masquée]**

Adaptation du TP AE / VAE / CVAE / PixelCNN à **CIFAR-10**, selon la demande orale
rapportée par l'étudiant. « CIFAR dix » est l'interprétation retenue du nom entendu.
Le PDF d'origine utilise FashionMNIST et MNIST ; cette version utilise CIFAR-10
pour les quatre blocs. Les images restent en couleur, sans conversion en gris.

Lire de haut en bas : **objectif → code commenté → sortie → explication**.
Les classes Python ne sont utilisées que pour définir les réseaux PyTorch.
Les boucles d'apprentissage restent visibles et explicites.

In [ ]:
# Importer uniquement les bibliothèques utilisées dans ce notebook.
from pathlib import Path
import json
import time
import random
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Fixer les tirages pour pouvoir refaire une expérience comparable.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(6)

# Utiliser le GPU si ce PC en possède un ; sinon utiliser le processeur.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Calcul sur :", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Tous les fichiers créés restent dans le dossier de ce TP.
racine = Path.cwd()
(racine / "figures").mkdir(exist_ok=True)
(racine / "models").mkdir(exist_ok=True)
debut = time.time()

## Charger les images couleur
Une image contient **3 × 32 × 32 = 3 072 nombres** : rouge, vert et bleu.
La division par 255 place les intensités dans [0,1]. Le test est chargé séparément
et ne sert jamais à ajuster les poids. Les classes ne supervisent que le CVAE.

In [ ]:
# Télécharger les vraies images CIFAR-10 avec torchvision.
# Au prochain lancement, les fichiers locaux sont réutilisés.
train = CIFAR10(root="data", train=True, download=True)
test = CIFAR10(root="data", train=False, download=True)

# CIFAR stocke [image, hauteur, largeur, couleur].
# PyTorch attend [image, couleur, hauteur, largeur].
x_train = torch.tensor(train.data).permute(0, 3, 1, 2).float() / 255.0
x_test = torch.tensor(test.data).permute(0, 3, 1, 2).float() / 255.0
y_train = torch.tensor(train.targets)
y_test = torch.tensor(test.targets)

# Former des lots de 128 images pour l'apprentissage.
donnees_train = TensorDataset(x_train, y_train)
chargeur = DataLoader(donnees_train, batch_size=128, shuffle=True)
classes = ["avion", "voiture", "oiseau", "chat", "cerf",
           "chien", "grenouille", "cheval", "bateau", "camion"]
print("Apprentissage :", x_train.shape)
print("Test :", x_test.shape)
print("Classes :", classes)
assert x_train.shape == (50000, 3, 32, 32)
assert x_test.shape == (10000, 3, 32, 32)

In [ ]:
# Cette fonction sert seulement à afficher et sauvegarder une grille de couleurs.
# permute remet les couleurs en dernier, comme attendu par matplotlib.
def afficher(images, titre, fichier, colonnes=8):
    images = images.detach().cpu().clamp(0, 1)
    lignes = (len(images) + colonnes - 1) // colonnes
    figure, axes = plt.subplots(lignes, colonnes,
                               figsize=(colonnes * 1.4, lignes * 1.5),
                               squeeze=False)
    for numero, axe in enumerate(axes.flat):
        axe.axis("off")
        if numero < len(images):
            axe.imshow(images[numero].permute(1, 2, 0))
    figure.suptitle(titre)
    figure.tight_layout()
    chemin = racine / "figures" / fichier
    figure.savefig(chemin, dpi=140, bbox_inches="tight")
    plt.close(figure)
    display(Image(filename=str(chemin)))

# Une seconde petite fonction affiche une courbe déjà calculée.
def courbe(valeurs, titre, fichier):
    plt.figure(figsize=(6, 3))
    plt.plot(range(1, len(valeurs) + 1), valeurs, marker="o")
    plt.xlabel("Époque")
    plt.ylabel("Perte moyenne")
    plt.title(titre)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    chemin = racine / "figures" / fichier
    plt.savefig(chemin, dpi=140)
    plt.close()
    display(Image(filename=str(chemin)))

## Construire le modèle probabiliste
μ est la moyenne et logvar le logarithme de variance. Le prior est N(0,I).
Les couches et dimensions restent explicites. La sortie contient trois canaux RGB.

In [ ]:
# Le VAE produit une moyenne et un logarithme de variance, pas un code unique.
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encodeur = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), nn.Flatten()
        )
        self.moyenne = nn.Linear(64 * 8 * 8, 32)
        self.log_variance = nn.Linear(64 * 8 * 8, 32)
        self.decodeur = nn.Sequential(
            nn.Linear(32, 64 * 8 * 8), nn.ReLU(), nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )

    def encoder(self, x):
        h = self.encodeur(x)
        return self.moyenne(h), self.log_variance(h)

    # Tirer z = moyenne + écart-type × bruit, tout en gardant les gradients.
    def reparametrer(self, mu, logvar):
        sigma = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(mu)
        return mu + sigma * epsilon

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparametrer(mu, logvar)
        return self.decodeur(z), mu, logvar

## Comprendre la perte
La BCE mesure la reconstruction sur des intensités dans [0,1]. La KL rapproche
les distributions latentes de N(0,I). Le coefficient β est fixé à 1.
Nous n'effectuons pas l'expérience β-VAE optionnelle.

In [ ]:
# Somme des erreurs par image, puis moyenne sur le lot, pour les deux termes.
def vae_loss(reconstruction, images, mu, logvar):
    bce = F.binary_cross_entropy(reconstruction, images, reduction="sum") / len(images)
    kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(images)
    total = bce + kl
    return total, bce, kl

In [ ]:
# Optimiser la reconstruction ET la régularisation du code latent.
modele = VAE().to(device)
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.001)
historique = []
for epoque in range(15):
    modele.train()
    total_bce = 0.0
    total_kl = 0.0
    for images, etiquettes in chargeur:
        images = images.to(device)
        etiquettes = etiquettes.to(device)
        optimiseur.zero_grad()
        reconstruction, mu, logvar = modele(images)
        erreur, bce, kl = vae_loss(reconstruction, images, mu, logvar)
        # Calculer les gradients, puis modifier les poids.
        erreur.backward()
        optimiseur.step()
        total_bce += bce.item() * len(images)
        total_kl += kl.item() * len(images)
    ligne = {"BCE": total_bce / len(x_train), "KL": total_kl / len(x_train)}
    historique.append(ligne)
    print("VAE", "époque", epoque + 1, ligne, flush=True)
modele.eval()
torch.save(modele.state_dict(), racine / "models" / "VAE.pt")

In [ ]:
# Séparer la reconstruction et la KL pour comprendre ce que le VAE optimise.
plt.figure(figsize=(6, 3))
plt.plot([ligne["BCE"] for ligne in historique], label="BCE")
plt.plot([ligne["KL"] for ligne in historique], label="KL")
plt.xlabel("Époque (indice à partir de 0)")
plt.ylabel("Perte moyenne par image")
plt.legend()
plt.tight_layout()
chemin = racine / "figures" / "B_pertes.png"
plt.savefig(chemin, dpi=140)
plt.close()
display(Image(filename=str(chemin)))

## Reconstruire puis générer : deux opérations différentes
Pour reconstruire, on encode une image test et décode μ.
Pour générer, on tire un nouveau z ; aucune image originale n'est fournie.

In [ ]:
# Reconstructions avec le code moyen, pour stabiliser l'affichage.
with torch.no_grad():
    mu, logvar = modele.encoder(x_test[:8].to(device))
    reconstructions = modele.decodeur(mu).cpu()
afficher(torch.cat([x_test[:8], reconstructions]),
         "Originaux / VAE", "B_reconstructions.png")

# Générations libres à partir du prior gaussien.
with torch.no_grad():
    z = torch.randn(32, 32, device=device)
    generations = modele.decodeur(z).cpu()
afficher(generations, "VAE : nouvelles images RGB", "B_generations.png")

In [ ]:
# Interpoler entre deux codes aléatoires du prior.
interpolations = []
with torch.no_grad():
    for alpha in torch.linspace(0, 1, 8):
        code_latent = (1 - alpha) * z[0] + alpha * z[1]
        image = modele.decodeur(code_latent.unsqueeze(0))
        interpolations.append(image.cpu())
afficher(torch.cat(interpolations), "VAE : interpolation", "B_interpolation.png")

In [ ]:
# Projeter les moyennes de 1 000 images test ; les étiquettes colorent seulement le nuage.
from sklearn.manifold import TSNE
codes = []
with torch.no_grad():
    for images in x_test[:1000].split(128):
        mu, logvar = modele.encoder(images.to(device))
        codes.append(mu.cpu())
codes = torch.cat(codes).numpy()
projection = TSNE(n_components=2, perplexity=30, random_state=42,
                  init="pca", learning_rate="auto").fit_transform(codes)
plt.figure(figsize=(6, 4))
points = plt.scatter(projection[:, 0], projection[:, 1], c=y_test[:1000], cmap="tab10", s=8)
plt.colorbar(points, ticks=range(10))
plt.title("VAE : moyennes latentes projetées")
plt.tight_layout()
chemin = racine / "figures" / "B_tsne.png"
plt.savefig(chemin, dpi=140)
plt.close()
display(Image(filename=str(chemin)))

## Réponses aux trois questions B
1. La KL rapproche les distributions encodées d'un même prior : les tirages depuis
ce prior ont plus de sens qu'avec un AE non régularisé.
2. Une KL forte limite l'information transmise et peut dégrader la reconstruction ;
une KL faible peut laisser des régions du prior mal apprises.
3. Les sorties peuvent être floues car le modèle distribue son incertitude entre
plusieurs détails plausibles. CIFAR-10 est visuellement plus complexe que MNIST.
Ne pas confondre une image colorée avec un objet clairement reconnaissable.

In [ ]:
# Enregistrer les pertes réelles, sans inventer de score de qualité visuelle.
resultats = {"historique": historique, "latent": 32, "epochs": 15,
             "secondes": time.time() - debut}
(racine / "resultats_B.json").write_text(json.dumps(resultats, indent=2), encoding="utf-8")
print("Bloc B terminé.")